In [ ]:
import os
from pathlib import Path

BASE_DIR = Path.cwd()
ROOT_DIR = BASE_DIR.parent
CACHE_DIR = str(ROOT_DIR.parent / "huggingface_cache")

os.environ["HF_HOME"] = CACHE_DIR
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import random
import json
from tqdm import tqdm
import torch
from dotenv import load_dotenv
from transformers import set_seed
from unsloth import FastLanguageModel, get_chat_template
load_dotenv()

SEED = 42
set_seed(SEED)
random.seed(SEED)

BATCH_SIZE = 8

sys.path.insert(0, str(ROOT_DIR))

DATA_DIR = (ROOT_DIR / "data/sft_data/qna.json").resolve()
MODEL_DIR = (ROOT_DIR / "models").resolve()

/tmp/ipykernel_3427502/519796475.py:18: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, get_chat_template


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
import unsloth
from unsloth import FastLanguageModel, get_chat_template

def get_model_and_tokenizer():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/Qwen3-8B",
        max_seq_length=2048,
        load_in_4bit=True,
    )
    tokenizer = get_chat_template(tokenizer, chat_template="qwen-3")
    return model, tokenizer

In [ ]:
model, tokenizer = get_model_and_tokenizer()

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files=str(DATA_DIR),
    split="train"
).shuffle(seed=SEED)

dataset

In [ ]:
def generate_conversation(examples):
    problems  = examples["instruction"]
    solutions = examples["output"]
    conversations = []
    for problem, solution in zip(problems, solutions):
        conversations.append([
            {"role" : "user",      "content" : problem},
            {"role" : "assistant", "content" : solution},
        ])
    return { "conversations": conversations, }

In [ ]:
dataset = dataset.map(generate_conversation, batched=True)

def format_chat(batch):
    texts = tokenizer.apply_chat_template(
        batch["conversations"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False
    )
    texts = [t + tokenizer.eos_token for t in texts]

    return {"text": texts}

dataset = dataset.map(format_chat, batched=True)
dataset = dataset.remove_columns(["instruction", "output", "conversations"])

In [ ]:
dataset.to_json("data.json")

Creating json from Arrow format:   0%|          | 0/20 [00:00<?, ?ba/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from unsloth import UnslothTrainer, UnslothTrainingArguments, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

class CustomTrainer(UnslothTrainer):
    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        inputs = self._prepare_inputs(inputs)

        with self.compute_loss_context_manager():
            loss = self.compute_loss(
                model, inputs, num_items_in_batch=num_items_in_batch
            )

        if not torch.isfinite(loss):
            print(f"[CustomTrainer] Skip step {self.state.global_step} (NaN/Inf)")

            self.accelerator.clear_gradients()

            return torch.zeros((), device=loss.device, dtype=loss.dtype)

        if self.args.n_gpu > 1:
            loss = loss.mean()

        self.accelerator.backward(loss)

        return loss.detach() / self.args.gradient_accumulation_steps


training_args = UnslothTrainingArguments(
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,

    warmup_ratio=0.1,
    num_train_epochs=2,
    learning_rate=1e-5,
    max_grad_norm=1.0,

    eval_strategy="steps",
    eval_steps=100,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,

    report_to="wandb",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=SEED,

    output_dir=MODEL_DIR / "sft1",
    gradient_checkpointing=False,
)

trainer = CustomTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=8,
    packing=False,
    args=training_args,
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

In [ ]:
tokenizer.decode(trainer.train_dataset[10]["input_ids"])

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

In [ ]:
# trainer.train()

In [ ]:
questions = [

# ================= VI - MCQ =================
"""Trả lời nhanh: Biên độ của sóng điện từ thay đổi như thế nào khi nó truyền qua một vật dẫn?

A. Nó vẫn không đổi.
B. Nó tăng lên.
C. Nó giảm đi.
D. Nó dao động""",

# ================= VI - QNA =================
"""Hãy trả lời câu hỏi sau:

Mục đích của tính cước ngoại tuyến (offline charging) là gì trong 3GPP?

Trả lời ngắn gọn và giải thích.""",

# ================= EN - MCQ =================
"""Quick answer: How are voice calls delivered to dual/multi-mode terminals in hybrid networks? [3GPP Release 18]

1. Based on available radio access systems
2. Based on serving network conditions
3. Based on roaming agreements between operators
4. Based on CS core network domain
5. Based on service delivery options in the user profile""",

# ================= EN - QNA =================
"""Answer the following question:

What is the purpose of Tracking Area Update (TAU) in LTE networks?

Provide a concise explanation."""
]


messages_list = [
    [{"role": "user", "content": q}]
    for q in questions
]

In [ ]:
from transformers import TextStreamer

for messages in messages_list:
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    print("\n===== =====\n")

    _ = model.generate(
        **tokenizer(text, return_tensors="pt").to("cuda"),
        max_new_tokens=512,
        temperature=0.5,
        top_p=0.8,
        top_k=20,
        streamer=TextStreamer(tokenizer, skip_prompt=False),
    )